# 2. Feature Engineering & Preprocessing

In this notebook, we transform the raw data into a mathematically sound state for our XGBoost model.
We will perform:
1. **Structural Cleaning**: Mapping hardware sentinels to NaN (XGBoost handles missing data natively).
2. **Feature Engineering**: Creating advanced hydrology features (like TWI and Runoff Acceleration) to capture flash flood physics.
3. **Data Splitting**: Stratified splitting to perfectly preserve the 8.4% class imbalance ratio.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv('synthetic_flood_data.csv')
print("Dataset loaded. Shape:", df.shape)
df.head()

Dataset loaded. Shape: (50000, 13)


,Rain_1h_mm,Rain_3h_mm,Rain_24h_mm,Forecast_Rain_3h_mm,Soil_Moisture_Pct,Slope_Deg,Elevation_m,Distance_to_River_m,Land_Cover_Type,Flow_Accumulation,River_Water_Level_m,Risk_Score_Raw,Flood_Risk_Label
0,0.07,7.59,12.68,21.52,17.37,20.49,908.03,405.78,1,70.54,1.14,72.45,1
1,0.89,28.69,39.15,57.18,54.18,3.72,2461.80,93.80,3,53.24,1.44,80.82,2
2,3.96,5.42,17.16,56.93,62.75,7.67,1082.43,663.71,2,41.09,1.31,57.76,1
3,1.14,7.40,28.18,26.25,52.33,35.29,158.99,1933.63,2,33.81,1.12,66.80,1
4,1.00,11.87,79.56,0.95,81.34,46.66,2947.39,1934.43,1,63.69,2.17,75.10,1


## 2.1 Structural Cleaning
We map the sentinel values (-999.0 and 9999.9) to NaN. Crucially, we **do not impute** these values. We leave them as NaN because XGBoost uses 'Sparsity-Aware Splitting' to learn natural patterns from missing data.

In [17]:
# Map Sentinels to NaN (Do NOT impute! XGBoost handles NaNs natively)
clean_df = df.drop(columns=['Risk_Score_Raw'])
clean_df.loc[clean_df['River_Water_Level_m'] == -999.0, 'River_Water_Level_m'] = np.nan
clean_df.loc[clean_df['Rain_1h_mm'] == 9999.9, 'Rain_1h_mm'] = np.nan
clean_df.loc[clean_df['Soil_Moisture_Pct'] == 0.0, 'Soil_Moisture_Pct'] = np.nan


## 2.2 Feature Engineering (Physics Interactions)
We are injecting domain-specific hydrology physics into the dataset. By combining raw variables into advanced metrics (like the Topographic Wetness Index and Storm Trend), we mathematically hand the AI the laws of physics, making the model faster, smarter, and significantly more accurate.

In [18]:
# --------------------------------------------------------
# THE ULTIMATE PHYSICS ENGINE 
# --------------------------------------------------------
import numpy as np

# 1. Sentinels to NaN (MUST HAPPEN BEFORE MATH)
clean_df['Is_Rain_Sentinel'] = clean_df['Rain_1h_mm'].isna().astype(int)
clean_df['Is_River_Sentinel'] = clean_df['River_Water_Level_m'].isna().astype(int)
clean_df['Is_Forecast_Sentinel'] = clean_df['Forecast_Rain_3h_mm'].isna().astype(int)

# 2. TWI: Topographic Wetness Index (Hydrology Standard)
slope_rad = np.radians(clean_df['Slope_Deg'].clip(lower=0.1))
clean_df['TWI'] = np.log((clean_df['Flow_Accumulation'] + 1) / np.tan(slope_rad))

# 3. Topography Correction (Steep terrain NEAR a river)
clean_df['River_Proximity_Score'] = 1 / (clean_df['Distance_to_River_m'] + 10)
clean_df['Steepness_Danger'] = clean_df['Slope_Deg'] * clean_df['River_Proximity_Score']

# 4. Temporal Rainfall Structure 
clean_df['Storm_Trend'] = clean_df['Forecast_Rain_3h_mm'] / (clean_df['Rain_3h_mm'] + 1)
clean_df['Rain_Burst_Ratio'] = clean_df['Rain_1h_mm'] / (clean_df['Rain_3h_mm'] + 1)

# 5. Soil & Runoff Dynamics ()
clean_df['Soil_Deficit'] = 100 - clean_df['Soil_Moisture_Pct']
clean_df['Runoff_Acceleration'] = clean_df['Rain_1h_mm'] / (clean_df['Soil_Deficit'] + 10)

# 6. River Threat Interactions
clean_df['River_Rain1'] = clean_df['River_Water_Level_m'] * clean_df['Rain_1h_mm']
clean_df['Doorstep_Threat'] = clean_df['River_Water_Level_m'] * clean_df['River_Proximity_Score']

# 7. Safety Catch: Kill Infs
clean_df = clean_df.replace([np.inf, -np.inf], np.nan)

# 8. Drop Flawed/Redundant Standalone Columns to prevent Feature Importance splitting 
if 'Gravity_Danger' in clean_df.columns: clean_df = clean_df.drop(columns=['Gravity_Danger'])

print("Master Features Created successfully.")
display(clean_df[['TWI', 'Steepness_Danger', 'Doorstep_Threat', 'Runoff_Acceleration']].head())


Master Features Created successfully.


,TWI,Steepness_Danger,Doorstep_Threat,Runoff_Acceleration
0,5.254597,0.049281,0.002742,0.000756
1,6.726515,0.035838,0.013873,0.015944
2,5.744722,0.011385,0.001944,0.083810
3,3.895530,0.018157,0.000576,0.019768
4,4.111629,0.023997,0.001116,0.034892


> **[FEATURE ENGINEERING INSIGHTS]**
> By combining individual variables into physics-based interaction features, we provide baseline models a direct mathematical pathway to understand flash flood mechanics, significantly boosting their initial accuracy.

## 2.3 Train-Test Split (Stratified)
Because our data is heavily imbalanced (Only 8.4% Evacuate class), a random split might accidentally put 0% of the Evacuate cases in the test set. We MUST use `stratify=y` to lock the 8.4% ratio perfectly across both training and testing sets.

In [19]:
X = clean_df.drop(columns=['Flood_Risk_Label'])
y = clean_df['Flood_Risk_Label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train size: {X_train.shape[0]} rows")
print(f"Test size: {X_test.shape[0]} rows")

Train size: 40000 rows
Test size: 10000 rows


## 2.5 Exporting Pipeline Artifacts
We now save our processed `X_train`, `X_test`, `y_train`, and `y_test` datasets so that `03_Model_Building.ipynb` can load them perfectly clean.

In [20]:
os.makedirs('processed_data', exist_ok=True)

X_train.to_csv('processed_data/X_train.csv', index=False)
X_test.to_csv('processed_data/X_test.csv', index=False)
y_train.to_csv('processed_data/y_train.csv', index=False)
y_test.to_csv('processed_data/y_test.csv', index=False)

print("Data exported to /processed_data/ successfully!")

Data exported to /processed_data/ successfully!


## 2.6 Conclusion: Pipeline Output & Readiness

We have transformed the raw dataset into a highly optimized, physics-informed state ready for tree-based modeling. Summary:

1. **Sentinel Preservation:** We safely mapped hardware errors to NaN without artificially imputing them, allowing our future model to learn from real-world missingness.
2. **Advanced Hydrology Features:** We engineered critical domain-specific variables, including the Topographic Wetness Index (TWI), Storm Trend, and Steepness Danger.
3. **Stratified Integrity:** We split the data into 80/20 sets using stratify=y. This guarantees that the critical Class 3 label maintains its exact representation.
4. **Export:** We successfully exported X_train.csv and others to the processed_data directory.

> **Next Steps:** The data is now perfectly formatted. In the Model Building notebook, we will load these artifacts and train our predictive engines.